# Notebook 2: Magnitude, Homology, and Perturbation Theory

**Paper:** *Measuring What Persists: Magnitude Homology as a Structural Fingerprint for LLM Identity Drift*

This notebook verifies every mathematical computation in the paper, starting from the distance matrices. It covers:
- **A.** Scalar magnitude computation (§5.2, §5.3, §5.5, §5.6)
- **B.** Eigenvalue analysis (§2.3, Appendix A.1–A.2)
- **C.** Perturbation bounds (Appendix A.1)
- **D.** First-order perturbation predictions (Appendix A.2–A.3) — the 0.19% and 0.11% agreement results
- **E.** Second-order shape contributions (Appendix A.3.1)
- **F.** Mode decomposition — breathing vs shearing (Appendix A.5)
- **G.** Betweenness and MH groups (§5.2, §5.4, §5.5, §5.6)

**Teaching moment:** The magnitude computation is a 3×3 linear solve. The perturbation theory is a first-order matrix expansion. Everything here can be verified with pencil, paper, and a calculator.

In [1]:
import json
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")
drift = json.loads((DATA_DIR / "drift_experiment_2026-05-06.json").read_text())

SQRT_LN2 = np.sqrt(np.log(2))
d = SQRT_LN2  # equilateral baseline distance
a = np.exp(-d)  # off-diagonal similarity entry

probes = ["Q1", "Q3", "B4"]

def get_D(condition, cond_type):
    dm = drift["distance_matrices"][condition][cond_type]
    D = np.zeros((3, 3))
    for i, pi in enumerate(probes):
        for j, pj in enumerate(probes):
            D[i, j] = dm[pi][pj]
    return D

print(f"d = √(ln 2) = {d:.10f}")
print(f"a = e^(-d) = {a:.10f}")

d = √(ln 2) = 0.8325546112
a = e^(-d) = 0.4349367716


## A. Scalar magnitude computation

For a finite metric space with distance matrix $D$, the similarity matrix is $Z_{ij} = e^{-d_{ij}}$. The magnitude is $|M| = \mathbf{1}^T Z^{-1} \mathbf{1}$ — the sum of all entries of $Z^{-1}$. Equivalently: solve $Z\mathbf{w} = \mathbf{1}$ and sum the weights $\mathbf{w}$.

In [2]:
def scalar_magnitude(D):
    Z = np.exp(-D)
    w = np.linalg.solve(Z, np.ones(D.shape[0]))
    return float(np.sum(w)), w

# Compute magnitude for all conditions
print("Scalar magnitude |M| for all conditions\n")
print(f"{'Condition':<25} {'|M|':>10} {'Paper':>10} {'w_Q1':>8} {'w_Q3':>8} {'w_B4':>8}")
print("-" * 75)

for cond in ["baseline", "medium", "long"]:
    for ct, ct_label in [("card", "card"), ("base", "base")]:
        D = get_D(cond, ct)
        mag, w = scalar_magnitude(D)
        label = f"{cond} / {ct_label}"
        
        if ct == "card":
            paper_mag = {
                "baseline": 1.6044, "medium": 1.5875, "long": 1.5888
            }[cond]
        else:
            paper_mag = 1.6044
        
        print(f"{label:<25} {mag:>10.4f} {paper_mag:>10.4f} {w[0]:>8.4f} {w[1]:>8.4f} {w[2]:>8.4f}")

print(f"\nBaseline |M| = 1.6044: this is the equilateral reference.")
print(f"Medium  |M| = 1.5875: magnitude decreased — drift detected.")
print(f"Long    |M| = 1.5888: magnitude decreased — drift detected.")

Scalar magnitude |M| for all conditions

Condition                        |M|      Paper     w_Q1     w_Q3     w_B4
---------------------------------------------------------------------------
baseline / card               1.6044     1.6044   0.5348   0.5348   0.5348
baseline / base               1.6044     1.6044   0.5348   0.5348   0.5348
medium / card                 1.5875     1.5875   0.5318   0.5362   0.5194
medium / base                 1.6044     1.6044   0.5348   0.5348   0.5348
long / card                   1.5888     1.5888   0.5324   0.5243   0.5321
long / base                   1.6044     1.6044   0.5348   0.5348   0.5348

Baseline |M| = 1.6044: this is the equilateral reference.
Medium  |M| = 1.5875: magnitude decreased — drift detected.
Long    |M| = 1.5888: magnitude decreased — drift detected.


## B. Eigenvalue analysis

For the equilateral similarity matrix $Z = (1-a)I + aJ$, the eigenvalues follow from the rank-1 structure of $J$:
- $\lambda_1 = 1 + 2a$ (eigenvector $\mathbf{1}$, multiplicity 1)
- $\lambda_2 = \lambda_3 = 1 - a$ (orthogonal to $\mathbf{1}$, multiplicity 2)

The inverse has the analytic form $Z^{-1} = \frac{1}{1-a}I - \frac{a}{(1-a)(1+2a)}J$.

In [3]:
# Eigenvalues
lambda1 = 1 + 2*a
lambda2 = 1 - a
kappa = lambda1 / lambda2

print("Eigenvalue analysis\n")
print(f"λ₁ = 1 + 2a = {lambda1:.4f}  (eigenvector 1, multiplicity 1)")
print(f"λ₂ = λ₃ = 1 - a = {lambda2:.4f}  (orthogonal to 1, multiplicity 2)")
print(f"κ(Z) = λ₁/λ₂ = {kappa:.4f}")
print(f"\nBoth positive and bounded away from zero. Singular locus at d=0 only.")

# Verify numerically
D_eq = np.array([[0, d, d], [d, 0, d], [d, d, 0]])
Z_eq = np.exp(-D_eq)
eigs = np.linalg.eigvalsh(Z_eq)
print(f"\nNumerical eigenvalues: {sorted(eigs)}")
print(f"Match: λ₂={eigs[0]:.4f} vs {lambda2:.4f}, λ₁={eigs[2]:.4f} vs {lambda1:.4f}")

# Analytic Z⁻¹
I = np.eye(3)
J = np.ones((3, 3))
Z_inv_analytic = (1/(1-a)) * I - (a / ((1-a)*(1+2*a))) * J
Z_inv_numeric = np.linalg.inv(Z_eq)

print(f"\nZ⁻¹ analytic vs numeric max error: {np.max(np.abs(Z_inv_analytic - Z_inv_numeric)):.2e}")
print(f"Z⁻¹·1 = (1/(1+2a))·1 = {1/(1+2*a):.4f}·1")
print(f"Verify: Z⁻¹·1 = {Z_inv_analytic @ np.ones(3)}")

Eigenvalue analysis

λ₁ = 1 + 2a = 1.8699  (eigenvector 1, multiplicity 1)
λ₂ = λ₃ = 1 - a = 0.5651  (orthogonal to 1, multiplicity 2)
κ(Z) = λ₁/λ₂ = 3.3091

Both positive and bounded away from zero. Singular locus at d=0 only.

Numerical eigenvalues: [np.float64(0.56506322842429), np.float64(0.5650632284242901), np.float64(1.8698735431514202)]
Match: λ₂=0.5651 vs 0.5651, λ₁=1.8699 vs 1.8699

Z⁻¹ analytic vs numeric max error: 5.55e-17
Z⁻¹·1 = (1/(1+2a))·1 = 0.5348·1
Verify: Z⁻¹·1 = [0.53479552 0.53479552 0.53479552]


## C–D. Perturbation bounds and first-order predictions

The first-order perturbation formula (Eq. A.7 in the paper): $\delta|M| = \frac{2a}{(1+2a)^2} \cdot \delta P$

where $\delta P$ is the perimeter change of the probe triangle. No free parameters. The coefficient $\frac{2a}{(1+2a)^2} \approx 0.249$ is determined entirely by the equilateral geometry.

**Why the worst-case bound overestimates:** The bound uses $1/\lambda_{\min} = 1/(1-a) \approx 1.77$, but magnitude responds only to the $\mathbf{1}$-eigenvector direction with eigenvalue $1/(1+2a) \approx 0.53$. The ratio is $(1-a)^2/(1+2a)^2 \approx 0.09$ — explaining the order-of-magnitude gap.

In [4]:
D_base = get_D("baseline", "card")
D_med = get_D("medium", "card")
D_long = get_D("long", "card")
mag_base = scalar_magnitude(D_base)[0]

# Perturbation coefficient
coeff = 2*a / (1 + 2*a)**2
print(f"Perturbation coefficient: 2a/(1+2a)² = {coeff:.4f}")
print(f"Coefficient ratio (actual/worst-case): (1-a)²/(1+2a)² = {(1-a)**2/(1+2*a)**2:.4f}")

# Distance changes
delta_d_med = np.array([D_med[0,1]-D_base[0,1], D_med[0,2]-D_base[0,2], D_med[1,2]-D_base[1,2]])
delta_d_long = np.array([D_long[0,1]-D_base[0,1], D_long[0,2]-D_base[0,2], D_long[1,2]-D_base[1,2]])

print(f"\n{'':>30} {'Q1-Q3':>10} {'Q1-B4':>10} {'Q3-B4':>10} {'δP':>10}")
print("-" * 70)
print(f"{'Medium δd':<30} {delta_d_med[0]:>10.4f} {delta_d_med[1]:>10.4f} {delta_d_med[2]:>10.4f} {sum(delta_d_med):>10.4f}")
print(f"{'Long δd':<30} {delta_d_long[0]:>10.4f} {delta_d_long[1]:>10.4f} {delta_d_long[2]:>10.4f} {sum(delta_d_long):>10.4f}")

# Predictions
for label, delta_d in [("Medium", delta_d_med), ("Long", delta_d_long)]:
    dP = sum(delta_d)
    predicted = coeff * dP
    D_drift = get_D(label.lower(), "card")
    observed = scalar_magnitude(D_drift)[0] - mag_base
    agreement = abs(predicted - observed) / abs(observed) * 100
    
    print(f"\n{label} context:")
    print(f"  Predicted δ|M| = {coeff:.4f} × {dP:.4f} = {predicted:.5f}")
    print(f"  Observed  δ|M| = {scalar_magnitude(D_drift)[0]:.4f} - {mag_base:.4f} = {observed:.5f}")
    print(f"  Agreement: {agreement:.2f}%")

# Worst-case bound
dZ_med = np.zeros((3,3))
for i in range(3):
    for j in range(3):
        if i != j:
            dZ_med[i,j] = -np.exp(-D_base[i,j]) * (D_med[i,j] - D_base[i,j])
norm_dZ = np.linalg.norm(dZ_med, ord=2)
bound = 3 * (1/(1-a))**2 * norm_dZ
obs_med = abs(scalar_magnitude(D_med)[0] - mag_base)
print(f"\nWorst-case bound: {bound:.3f}")
print(f"Observed change:  {obs_med:.3f}")
print(f"Ratio observed/bound: {obs_med/bound:.2f} (≈ 1/10)")

Perturbation coefficient: 2a/(1+2a)² = 0.2488
Coefficient ratio (actual/worst-case): (1-a)²/(1+2a)² = 0.0913

                                    Q1-Q3      Q1-B4      Q3-B4         δP
----------------------------------------------------------------------
Medium δd                         -0.0000    -0.0393    -0.0286    -0.0679
Long δd                           -0.0268    -0.0084    -0.0275    -0.0626

Medium context:
  Predicted δ|M| = 0.2488 × -0.0679 = -0.01690
  Observed  δ|M| = 1.5875 - 1.6044 = -0.01693
  Agreement: 0.19%

Long context:
  Predicted δ|M| = 0.2488 × -0.0626 = -0.01557
  Observed  δ|M| = 1.5888 - 1.6044 = -0.01559
  Agreement: 0.11%

Worst-case bound: 0.199
Observed change:  0.017
Ratio observed/bound: 0.09 (≈ 1/10)


## E–F. Second-order shape and mode decomposition

Any perturbation of the equilateral triangle decomposes into:
- **Breathing mode:** uniform expansion/contraction (changes all edges equally). Carries the entire perimeter change. Magnitude responds to this at first order.
- **Shearing mode:** shape distortion (redistributes distance among edges, preserving perimeter). First-order invisible to magnitude. Only enters at second order.

The equilateral configuration is a critical point of the magnitude functional on the constant-perimeter submanifold — shape changes are first-order cancelled.

**Teaching moment (bootstrap mean vs point estimate):** The bootstrap mean (1.5907) differs from the point estimate (1.5875) by the bootstrap bias. This positive bias means resampling with replacement from k=50 produces slightly higher mean magnitude. The point estimate is the single computation from all 50 samples; the bootstrap mean is the average over 1,000 resampled computations. The paper reports bootstrap means in Table 6 — not point estimates.

In [5]:
for label, delta_d in [("Medium", delta_d_med), ("Long", delta_d_long)]:
    mean_dd = np.mean(delta_d)
    breathing = np.full_like(delta_d, mean_dd)
    shearing = delta_d - breathing
    
    print(f"=== {label} context ===")
    print(f"  δd = ({delta_d[0]:+.4f}, {delta_d[1]:+.4f}, {delta_d[2]:+.4f})")
    print(f"  mean δd = {mean_dd:.4f}")
    print(f"  Breathing: ({breathing[0]:+.4f}, {breathing[1]:+.4f}, {breathing[2]:+.4f})")
    print(f"  Shearing:  ({shearing[0]:+.4f}, {shearing[1]:+.4f}, {shearing[2]:+.4f})")
    
    # Verify algebraic identities
    breathing_perim = np.sum(breathing)
    shearing_perim = np.sum(shearing)
    total_perim = np.sum(delta_d)
    print(f"  Breathing perimeter: {breathing_perim:.6f} = total perimeter: {total_perim:.6f} ✓")
    print(f"  Shearing perimeter:  {shearing_perim:.2e} = 0 (exact) ✓")
    
    # Norm ratio (what the paper reports as "variance fraction")
    norm_ratio = np.sqrt(np.sum(shearing**2)) / np.sqrt(np.sum(breathing**2))
    print(f"  ||shearing|| / ||breathing|| = {norm_ratio:.3f} ({norm_ratio*100:.1f}%)")
    
    # Second-order shape term
    shape_coeff = np.exp(-2*d) / (1 + 2*np.exp(-d))**3
    shape_term = shape_coeff * np.sum(shearing**2)
    obs = abs(scalar_magnitude(get_D(label.lower(), "card"))[0] - mag_base)
    print(f"  2nd-order shape term: {shape_term:.6f}")
    print(f"  As % of 1st-order signal: {shape_term/obs*100:.2f}% (< 0.3%)")
    print()

=== Medium context ===
  δd = (-0.0000, -0.0393, -0.0286)
  mean δd = -0.0226
  Breathing: (-0.0226, -0.0226, -0.0226)
  Shearing:  (+0.0226, -0.0167, -0.0060)
  Breathing perimeter: -0.067936 = total perimeter: -0.067936 ✓
  Shearing perimeter:  -3.47e-18 = 0 (exact) ✓
  ||shearing|| / ||breathing|| = 0.733 (73.3%)
  2nd-order shape term: 0.000024
  As % of 1st-order signal: 0.14% (< 0.3%)

=== Long context ===
  δd = (-0.0268, -0.0084, -0.0275)
  mean δd = -0.0209
  Breathing: (-0.0209, -0.0209, -0.0209)
  Shearing:  (-0.0059, +0.0125, -0.0066)
  Breathing perimeter: -0.062599 = total perimeter: -0.062599 ✓
  Shearing perimeter:  -3.47e-18 = 0 (exact) ✓
  ||shearing|| / ||breathing|| = 0.424 (42.4%)
  2nd-order shape term: 0.000007
  As % of 1st-order signal: 0.04% (< 0.3%)



## G. Betweenness and contraction

The drift manifests as contraction — the triangle gets smaller — not collapse toward a geodesic. No betweenness relationships emerge at any epsilon tested. The triangle is metrically smaller but topologically unchanged.

In [6]:
print("Betweenness check (contraction vs collapse)\n")
for cond in ["baseline", "medium", "long"]:
    D = get_D(cond, "card")
    btw = drift["homology"][cond]["card"].get("betweenness", [])
    print(f"{cond}: {len(btw)} betweenness relationships")

# Triangle inequality gap at long context
D_long = get_D("long", "card")
sum_q1q3_q3b4 = D_long[0,1] + D_long[1,2]
d_q1_b4 = D_long[0,2]
print(f"\nLong context triangle inequality:")
print(f"  d(Q1,Q3) + d(Q3,B4) = {D_long[0,1]:.4f} + {D_long[1,2]:.4f} = {sum_q1q3_q3b4:.4f}")
print(f"  d(Q1,B4) = {d_q1_b4:.4f}")
print(f"  Gap = {sum_q1q3_q3b4 - d_q1_b4:.4f} (large — no geodesic collapse)")

# MH₁ rank splitting at medium
mh = drift["homology"]["medium"]["card"]["homology_ranks"]
print(f"\nMH₁ rank splitting at medium context:")
print(f"  ℓ=15: rank {mh['1']['15']}")
print(f"  ℓ=16: rank {mh['1']['16']}")
print(f"  (Baseline has clean rank 6 at a single ℓ bin — the splitting reflects")
print(f"   non-equilateral edges crossing filtration thresholds at adjacent bins.)")

Betweenness check (contraction vs collapse)

baseline: 0 betweenness relationships
medium: 0 betweenness relationships
long: 0 betweenness relationships

Long context triangle inequality:
  d(Q1,Q3) + d(Q3,B4) = 0.8058 + 0.8051 = 1.6109
  d(Q1,B4) = 0.8242
  Gap = 0.7867 (large — no geodesic collapse)

MH₁ rank splitting at medium context:
  ℓ=15: rank 2
  ℓ=16: rank 4
  (Baseline has clean rank 6 at a single ℓ bin — the splitting reflects
   non-equilateral edges crossing filtration thresholds at adjacent bins.)


## Summary

All mathematical computations verified:
- **Magnitude:** Baseline 1.6044, medium 1.5875, long 1.5888 — all match paper.
- **Eigenvalues:** λ₁ = 1.8699, λ₂ = 0.5651 (computed exactly; paper approximations contain a rounding error — see MSG-008 to Ada).
- **Perturbation coefficient:** 2a/(1+2a)² = 0.2488 — matches paper.
- **First-order predictions:** 0.19% and 0.11% agreement — the strongest theory-experiment connection in the paper.
- **Second-order shape terms:** < 0.3% of first-order signal — negligible.
- **Mode decomposition:** 73.3% shearing at medium — matches paper.
- **Betweenness:** Zero at all conditions — contraction, not collapse.
- **MH₁ splitting:** Rank 2 at ℓ=15, rank 4 at ℓ=16 — matches paper.